# Biomedical LLM adaptation — Colab T4 runner (5K slice)

Runs the **5K** data-scaling slice end-to-end on a **free Colab T4**: clone → install pinned deps → mount Drive → train → eval (base 0/5-shot + fine-tuned) → error analysis → headline table.

**Resumable:** all outputs live on Google Drive, and training resumes from the latest checkpoint. If Colab disconnects, just re-run from the top — you lose at most `save_steps` of progress.

**T4 notes:** Turing (compute 7.5) → fp16 (no bf16); 5K and 20K fit one free session, 50K/full do not. vLLM is skipped on T4 (recorded as not-feasible).

> Set Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
# 1. Verify the GPU
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Clone the repo
REPO_URL = 'https://github.com/Udit013/biomed-llm-peft.git'
import os
if not os.path.isdir('biomed-llm-peft'):
    !git clone $REPO_URL biomed-llm-peft
%cd biomed-llm-peft

In [ ]:
# 3. Install deps. Colab ships a CUDA-matched torch — do NOT reinstall it.
#    bitsandbytes MUST match Colab's torch/CUDA: pinning an old bnb breaks on
#    Colab's modern Triton ("No module named 'triton.ops'"), so we install the
#    latest bnb separately rather than the requirements pin.
!grep -vE '^(torch==|bitsandbytes)' requirements.txt > /tmp/reqs_core.txt
!pip install -q -r /tmp/reqs_core.txt
!pip install -q -U bitsandbytes        # latest wheel, matches Colab's CUDA build
# Verify bnb imported WITH GPU support (must print a version and find the CUDA .so):
!python -c "import bitsandbytes as bnb; print('bitsandbytes', bnb.__version__)"
print('deps installed')

In [ ]:
# 4. Mount Drive and redirect outputs/ + results/ there so they survive disconnects.
#    IMPORTANT: only the SMALL artifacts (adapters, eval JSON, logs) go to Drive.
#    Do NOT redirect the HF model cache (HF_HOME) to Drive — the 16 GB base model
#    won't fit a typical Drive and makes every run crawl. Let it cache to Colab's
#    fast local disk (re-downloads each fresh session; that's fine).
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/biomed-llm-peft'
for sub in ['outputs', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
    if os.path.islink(sub) or os.path.exists(sub):
        !rm -rf {sub}
    os.symlink(f'{DRIVE_ROOT}/{sub}', sub)
os.environ.pop('HF_HOME', None)   # ensure model cache stays on local disk, not Drive
print('outputs/ and results/ -> Drive:', DRIVE_ROOT)

In [ ]:
# 5. (Optional) experiment tracking + Hub. Leave blank to skip — training still runs.
import os
os.environ['WANDB_API_KEY'] = ''   # optional; blank => local logging only
os.environ['WANDB_PROJECT'] = 'biomed-llm-peft'
os.environ['HF_TOKEN'] = ''        # optional; needed only to push the adapter
!python -m src.utils.env

In [ ]:
# 6. Train the 5K slice (resumes from the latest Drive checkpoint if present).
!python scripts/train.py --config configs/qlora_5k.yaml

In [ ]:
# 7. Evaluate: base 0-shot, base 5-shot, and the fine-tuned adapter.
#    Full MedMCQA val (4183) + PubMedQA (1000) as 4-/3-way loglikelihood is ~20k
#    requests — too slow to finish a free-T4 session. For the 5K PIPELINE-
#    VALIDATION pass we score a fixed SUBSAMPLE (first N per task) so the whole
#    loop completes in-session and the tables get real numbers. Set
#    EVAL_LIMIT = None for the full official split (needs Colab Pro / a longer
#    or cloud GPU). lm-eval records the N actually scored in its results JSON.
#    Keep base + adapter at the SAME N so the comparison is apples-to-apples.
EVAL_LIMIT = 200                     # items per task; None = full official split
lim = f'--limit {EVAL_LIMIT}' if EVAL_LIMIT else ''
!python scripts/run_eval.py --mode base0 --output results/base_0shot $lim
!python scripts/run_eval.py --mode base5 --output results/base_5shot $lim
!python scripts/run_eval.py --adapter outputs/qlora_5k --output results/qlora_5k $lim

In [ ]:
# 8. Per-subject error analysis + headline table (real numbers from the runs above).
!python scripts/error_analysis.py --base results/base_0shot --finetuned results/qlora_5k --out results/error_analysis_qlora_5k
!python scripts/results_table.py --out results/headline_table.md
print(open('results/headline_table.md').read())

In [ ]:
# 9. Inference cost (quantized vs fp16 is the must-have on T4; vLLM auto-skips).
!python scripts/inference_cost.py --adapter outputs/qlora_5k --configs base lora_fp16 quantized vllm

In [ ]:
# 10. (Optional) push the adapter + model card to the Hub (needs HF_TOKEN above).
# !python scripts/push_to_hub.py --adapter outputs/qlora_5k --repo-id your-username/qwen2.5-7b-medmcqa-qlora-5k --base-eval results/base_0shot --ft-eval results/qlora_5k

All artifacts (adapters, eval JSON, logs, tables) are on Drive under `biomed-llm-peft/`. Pull them back to fill the README tables. To scale up, re-run cells 6–8 with `configs/qlora_20k.yaml` (still fits one free session).